In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine
import folium
import requests
import json
import re
from tqdm.auto import trange
import time
from requests.exceptions import JSONDecodeError

In [ ]:
geocode_api = 'my_api_key'

In [ ]:
# Create a world map zoomed in on NJ level 4
nj_map = folium.Map(location=[40.05832, -74.40566], zoom_start=7.5)

# Dsiplay the map
nj_map

In [ ]:
# Create a sql connection to import the sales data and town central lat and long data
tax_engine = create_engine(f"postgresql+psycopg2://postgres:pw@hostname:5433/nj_tax_assessor")  # Connection for sqlalchemy to get data

query = f'SELECT * FROM gsmls_imputed_data WHERE "YEAR" >= 2019;'
test_data = pd.read_sql_query(query, tax_engine)

In [ ]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423470 entries, 0 to 423469
Columns: 149 entries, MLSNUM to year_built
dtypes: bool(102), datetime64[ns](2), float64(23), int64(8), object(14)
memory usage: 193.0+ MB


In [ ]:
test_data.columns

Index(['MLSNUM', 'STATUS_SHORT', 'ADDRESS', 'TOWN', 'COUNTY', 'ZIPCODE',
       'NJ_TOWNCODE', 'TOWNCODE', 'COUNTYCODE', 'BLOCKID',
       ...
       'WATER_WATRXTRA', 'UTILITIES_ALLUNDER', 'UTILITIES_ELECTRIC',
       'UTILITIES_GASNATUR', 'UTILITIES_GASINSTR', 'UTILITIES_GASPROPN',
       'LISTING_REMARKS', 'TAX_PATTERN', 'building_sqft', 'year_built'],
      dtype='object', length=149)

In [ ]:
pd.set_option('display.max_columns', 999)

In [ ]:
# Add AGE_OF_PROPERTY column
target_cols = ['MLSNUM', 'STATUS_SHORT', 'ADDRESS', 'TOWN', 'COUNTY', 'ZIPCODE',
               'LISTPRICE', 'OLP/LP%', 'SALESPRICE', 'LISTDATE', 'CLOSEDDATE',  'year_built',
               'STYLEPRIMARY_SHORT', 'ROOMS', 'BEDS', 'BATHSTOTAL', 'building_sqft', 'LOTSIZE (SQFT)',
               'DAYSONMARKET', 'CONDITION', 'DISTRESSED_SALE', 'TAXAMOUNT', 'LATITUDE',
               'LONGITUDE']

In [ ]:
test_data = test_data[target_cols]

In [ ]:
test_data.rename(columns={'LATITUDE': 'PROP_LATITUDE', 'LONGITUDE': 'PROP_LONGITUDE'}, inplace=True)

In [ ]:
test_data.head()

,MLSNUM,STATUS_SHORT,ADDRESS,TOWN,COUNTY,ZIPCODE,LISTPRICE,OLP/LP%,SALESPRICE,LISTDATE,CLOSEDDATE,year_built,STYLEPRIMARY_SHORT,ROOMS,BEDS,BATHSTOTAL,building_sqft,LOTSIZE (SQFT),DAYSONMARKET,CONDITION,DISTRESSED_SALE,TAXAMOUNT,PROP_LATITUDE,PROP_LONGITUDE
0,1000032,SD,26 Woodward,Montgomery Twp,Somerset,08502,269900,-4.0,261500.0,1996-07-02,1996-12-19,NaN,Colonial,8.0,4.0,2.1,NaN,43560.0,140.0,Unknown,False,4762.0,40.448303,-74.644115
1,1000108,SD,27 Cedar Grove Lane,Franklin Twp,Somerset,08873,199000,0.0,190500.0,1996-05-21,1996-09-19,1978.0,Custom,8.0,4.0,2.0,2356.0,113256.0,64.0,Unknown,False,4106.0,40.531870,-74.511363
2,1000110,SD,21 Lebed Drive,Franklin Twp,Somerset,08873,164900,0.0,156000.0,1996-03-20,1996-10-07,1966.0,Develpmt,6.0,3.0,2.0,1900.0,14810.4,161.0,Unknown,False,3057.0,40.494261,-74.492888
3,1000145,SD,1408 Boxwood Trail,Branchburg Twp,Somerset,08876,144000,0.0,135250.0,1996-07-03,1996-07-19,1990.0,TwnEndUn,6.0,2.0,2.1,1642.0,43560.0,0.0,Unknown,False,2378.0,40.570645,-74.703989
4,1000190,SD,59 Highview Avenue,Bernardsville Boro,Somerset,07924,319719,-8.0,282000.0,1996-05-17,1997-05-12,1980.0,Colonial,8.0,4.0,2.1,1970.0,11325.6,328.0,Unknown,False,4333.0,40.724179,-74.566354


In [ ]:
test_data.loc[302145]

,302145
MLSNUM,2766651
STATUS_SHORT,SD
ADDRESS,38 Grant Ave
TOWN,Clifton City
COUNTY,Passaic
ZIPCODE,07011-3512
LISTPRICE,289900
OLP/LP%,0.0
SALESPRICE,250000.0
LISTDATE,2010-04-23 00:00:00


In [ ]:
test_sample = test_data.loc[302145]

In [ ]:
"""
I'll use the following guidelines to label comps and superior, eqyal and inferior
1) Location
  ***Use https://www.census.gov/data/academy/data-gems/2025/rural-or-urban-area.html
  - Label the area as urban, suburban or rural
  a. Comps in urban areas should be <= 0.5 miles
  b. Comps in suburban areas should be <= 1 mile
  c. Comps in rural areas should be <= 5 miles

2) Property Type
  - Only compare SFH to SFH or Condo to Condo

3) Size
  - Gross Living Area
  a. 0 - 1000 : +/- 25%
  b. 1001 - 2000 : +/- 20%
  c. 2001 - 3500 : +/- 15%
  d. 3501 - 5000 : +/- 10%
  e. 5000+ : +/- 10%

  - Lot size
  a. < 1 acre : +/- 30%
  b. 1-2.9 acres : +/- .5 acre
  c. 3-5.9 acres : +/- 1 acre
  d. 6-10.9 acres : +/- 2 acre
  e. 11+ acres : +/- 20%

4) Age
  a. 0 - 10 years : +/1 5 years
  b. 11 - 30 years : + 10 years/ -1/2 target age
  c. 30 - 50 years : +/15 years
  d. 51 - 75 years : +/20 years
  e. 75+ years : +/25 years

5) Physical Characteristics
  - Should have the same bedrooms and bathrooms
  - Quality changes +/-1 bedroom or bathroom?
  - Quality changes +/- 3 rooms
"""

In [ ]:
comp_properties = test_data.loc[(test_data['TOWN'] == test_sample['TOWN']) & (test_data['COUNTY'] == test_sample['COUNTY']) & (test_data['STATUS_SHORT'] == 'SD')
                                    & ( test_data['CLOSEDDATE'] <= test_sample['LISTDATE']) & (test_data['CLOSEDDATE'] > test_sample['LISTDATE'] - pd.Timedelta(days=90))]

In [ ]:
comp_properties.head()

,MLSNUM,STATUS_SHORT,ADDRESS,TOWN,COUNTY,ZIPCODE,LISTPRICE,OLP/LP%,SALESPRICE,LISTDATE,CLOSEDDATE,year_built,STYLEPRIMARY_SHORT,ROOMS,BEDS,BATHSTOTAL,building_sqft,LOTSIZE (SQFT),DAYSONMARKET,CONDITION,DISTRESSED_SALE,TAXAMOUNT,PROP_LATITUDE,PROP_LONGITUDE
264610,2563670,SD,192 Hazel St,Clifton City,Passaic,07011,210000,-37.0,193000.0,2008-07-01,2010-04-08,1920.0,Colonial,7.0,4.0,3.0,1287.0,3750.00,342.0,Unknown,False,5996.0,40.884379,-74.159485
269699,2664248,SD,24 West Parkway,Clifton City,Passaic,07014,899900,-8.0,775000.0,2009-03-18,2010-02-10,NaN,Custom,12.0,4.0,3.2,NaN,19602.00,237.0,Unknown,False,18365.0,40.843651,-74.143077
271279,2601930,SD,584 Clifton Avenue,Clifton City,Passaic,07011,190000,-46.0,192500.0,2008-11-12,2010-02-16,1926.0,Colonial,6.0,4.0,2.0,1214.0,3565.00,437.0,Unknown,True,6460.0,40.873323,-74.150952
276312,2707906,SD,28 Starmond Ave,Clifton City,Passaic,07013,294900,-6.0,275000.0,2009-08-20,2010-03-18,1929.0,Colonial,6.0,3.0,1.1,1492.0,3920.40,78.0,Unknown,False,7695.0,40.865152,-74.162735
276320,2708379,SD,115 Colfax Ave,Clifton City,Passaic,07013-1847,335000,-4.0,335000.0,2009-08-22,2010-01-26,1947.0,CapeCod,6.0,3.0,2.0,1540.0,6534.00,103.0,Unknown,False,6008.0,40.866672,-74.155050
277500,2715258,SD,12 Jaskot Ln,Clifton City,Passaic,07012-1105,349999,-5.0,330000.0,2009-09-19,2010-04-16,1955.0,CapeCod,7.0,4.0,1.1,1414.0,4791.60,163.0,Unknown,False,6817.0,40.853587,-74.150519
279594,2666794,SD,358 S Pky,Clifton City,Passaic,07014,350000,-12.0,315000.0,2009-03-23,2010-03-18,NaN,Colonial,7.0,3.0,1.1,NaN,6098.40,315.0,Unknown,False,7548.0,40.841577,-74.137393
281831,2669159,SD,544 Piaget Ave.,Clifton City,Passaic,07011,319000,-6.0,290000.0,2009-03-30,2010-03-29,1961.0,Bi-Level,9.0,4.0,2.0,1858.0,5400.00,164.0,Unknown,False,6934.0,40.878143,-74.158468
286945,2726778,SD,338 Mount Prospect Ave,Clifton City,Passaic,07012-1014,339000,0.0,316000.0,2009-11-09,2010-04-16,NaN,CapeCod,7.0,4.0,1.1,NaN,20808.00,129.0,Unknown,False,7885.0,40.856857,-74.160758
287428,2723738,SD,97 Sherwood St,Clifton City,Passaic,07013-1307,379000,0.0,350000.0,2009-10-26,2010-04-01,1948.0,Colonial,8.0,4.0,2.0,1536.0,4791.60,109.0,Unknown,False,8648.0,40.886766,-74.175695


In [ ]:
def create_popup(proprty_details: pd.Series):

  property_details = proprty_details.loc[["STATUS_SHORT", "CONDITION", "STYLEPRIMARY_SHORT", "ADDRESS", "LISTPRICE", "OLP/LP%", "SALESPRICE",
                                          "ROOMS", "BEDS", "BATHSTOTAL", "building_sqft", "CLOSEDDATE", "DAYSONMARKET"]]

  html = property_details.to_frame().to_html()

  popup = folium.Popup(folium.Html(html, script=True), max_width=500)

  return popup

In [ ]:
def query_geocode(address, town, zipcode):

  pass_list = ['Jersey City']
  remove_pattern = re.compile(r'Town|Twp|Boro|Village')

  if town not in pass_list:
    town = re.sub(remove_pattern, '', town)

  address = address.replace(' ', '+')
  town = town.replace(' ', '+')

  final_address = f'https://geocode.maps.co/search?q={address}+{town}NJ+{zipcode}+US&api_key={geocode_api}'

  json_obj = requests.get(final_address).json()

  if json_obj == []:
    return None
  else:
    return float(json_obj[0]['lat']), float(json_obj[0]['lon'])

In [ ]:
def create_address(address, town, zipcode):

  return f'{address}, {town}, NJ {zipcode}'

In [ ]:
from textwrap import fill

def create_base_map(property_details: pd.Series=None, default_location=[40.05832, -74.40566], default_zoom_start=7.5):

    radius_dict = {
        '0.25 miles': {'fill_color': 'green', 'fill_opacity': 0.5, 'radius': 402.336},
        '0.5 miles': {'fill_color': 'green', 'fill_opacity': 0.25, 'radius': 804.672},
        '1 mile': {'fill_color': 'blue', 'fill_opacity': 0.25, 'radius': 1609.344},
        '2 miles': {'fill_color': 'blue', 'fill_opacity': 0.20, 'radius': 3218.688}
    }

    property_group = folium.FeatureGroup(name='Properties')

    # Create a base map
    if property_details is None:
      base_map = folium.Map(location=default_location, zoom_start=default_zoom_start)
    else:
      # lat, lon = query_geocode(property_details['ADDRESS'], property_details['TOWN'], property_details['ZIPCODE'])
      lat, lon = property_details['PROP_LATITUDE'], property_details['PROP_LONGITUDE']
      base_map = folium.Map(location=[lat, lon], zoom_start=12)

      target_property_location = [lat, lon]
      popup = create_popup(property_details)

      property_group.add_child(
          folium.vector_layers.CircleMarker(
              location=target_property_location,
              tooltip=create_address(property_details['ADDRESS'], property_details['TOWN'], property_details['ZIPCODE']),
              popup=popup,
              radius=5, # Pixel radius
              # color='black', # Color of the outline of the shape
              stroke=False, # Controls if shape has an outline
              fill=True,
              fill_color='black',
              fill_opacity=0.7
          )
        )

      # Add a circle radius for nearest comps
      for key, args in radius_dict.items():
        folium.Circle(
            location=target_property_location,
            radius=args['radius'],
            tooltip=key,
            # color='black',
            fill_color=args['fill_color'],
            fill=True,
            stroke=False,
            fill_opacity=args['fill_opacity']
        ).add_to(base_map)

      return base_map, property_group

In [ ]:
def create_similar_comp(target_property: pd.Series, comp_property: pd.Series):

  # Create Boolean variables to guage target property and comp similarities

  room_diff = abs(target_property['ROOMS'] - comp_property['ROOMS'])
  sqft_diff = abs(target_property['building_sqft'] - comp_property['building_sqft'])
  similar_style = target_property['STYLEPRIMARY_SHORT'] == comp_property['STYLEPRIMARY_SHORT']
  similar_baths = target_property['BATHSTOTAL'] == comp_property['BATHSTOTAL']
  similar_rooms = room_diff < 3

  if 0 <= target_property['building_sqft'] <= 1000:
    similar_sqft = sqft_diff < (target_property['building_sqft'] * 0.25)

  elif 1001 <= target_property['building_sqft'] <= 2000:
    similar_sqft = sqft_diff < (target_property['building_sqft'] * 0.20)

  elif 2001 <= target_property['building_sqft'] <= 3500:
    similar_sqft = sqft_diff < (target_property['building_sqft'] * 0.15)

  elif 3501 <= target_property['building_sqft'] <= 5000:
    similar_sqft = sqft_diff < (target_property['building_sqft'] * 0.10)

  elif target_property['building_sqft'] >= 5001:
    similar_sqft = sqft_diff < (target_property['building_sqft'] * 0.10)

  popup = create_popup(comp_property)


  if similar_style is True:

    if (similar_baths is True) and (similar_rooms is True) and (similar_sqft is True):

      circle_marker = folium.vector_layers.CircleMarker(
                location=[comp_property['PROP_LATITUDE'], comp_property['PROP_LONGITUDE']],
                tooltip=create_address(comp_property['ADDRESS'], comp_property['TOWN'], comp_property['ZIPCODE']),
                popup=popup,
                radius=5,
                # color='black',
                stroke=False,
                fill=True,
                fill_color='blue',
                fill_opacity=0.7
            )

      return circle_marker

    else:

      circle_marker = folium.vector_layers.CircleMarker(
                location=[comp_property['PROP_LATITUDE'], comp_property['PROP_LONGITUDE']],
                tooltip=create_address(comp_property['ADDRESS'], comp_property['TOWN'], comp_property['ZIPCODE']),
                popup=popup,
                radius=5,
                # color='black',
                stroke=False,
                fill=True,
                fill_color='yellow',
                fill_opacity=0.7
            )

      return circle_marker

  else:

    circle_marker = folium.vector_layers.CircleMarker(
                location=[comp_property['PROP_LATITUDE'], comp_property['PROP_LONGITUDE']],
                tooltip=create_address(comp_property['ADDRESS'], comp_property['TOWN'], comp_property['ZIPCODE']),
                popup=popup,
                radius=5,
                # color='black',
                stroke=False,
                fill=True,
                fill_color='red',
                fill_opacity=0.7
            )

    return circle_marker

In [ ]:
def add_comp_properties(marker_group, proeprty_details: pd.Series=None):

  if proeprty_details is not None:

    town = proeprty_details['TOWN']
    county = proeprty_details['COUNTY']
    list_date = proeprty_details['LISTDATE']

    comp_properties = test_data.loc[(test_data['TOWN'] == town) & (test_data['COUNTY'] == county) & (test_data['STATUS_SHORT'] == 'SD')
                                    & ( test_data['CLOSEDDATE'] <= list_date) & (test_data['CLOSEDDATE'] > list_date - pd.Timedelta(days=90))]
    comp_properties = comp_properties.sort_values(by='LISTDATE', ascending=False)

    # if len(comp_properties) > 30:
    #   comp_properties = comp_properties.iloc[:30]

    for _, data in zip(trange(len(comp_properties)), comp_properties.iterrows()):
      row = data[1]

      try:
        # lat, lon = query_geocode(row['ADDRESS'], row['TOWN'], row['ZIPCODE'])
        marker_group.add_child(create_similar_comp(proeprty_details, row))
      except TypeError:
        pass

      except JSONDecodeError:
        pass

    return marker_group

In [ ]:
def create_comp_map(property_details: pd.Series=None):

    final_map, property_group = create_base_map(property_details)
    property_group = add_comp_properties(property_group, property_details)
    final_map.add_child(property_group)

    return final_map

In [ ]:
final_map = create_comp_map(test_data.loc[272861])

  0%|          | 0/42 [00:00<?, ?it/s]

In [ ]:
final_map